In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-30 12:15:24--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Распознаётся raw.githubusercontent.com (raw.githubusercontent.com)… 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Подключение к raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... соединение установлено.
HTTP-запрос отправлен. Ожидание ответа… 200 OK
Длина: 635180 (620K) [text/plain]
Сохранение в: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620,29K   853KB/s    за 0,7s    

2026-10-30 12:15:26 (853 KB/s) - ‘car_fuel_efficiency_2026.csv’ сохранён [635180/635180]



In [3]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [4]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
df.describe().round()

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.0,10000.0,10000.0,10000.0,9123.0,10000.0,9736.0,10000.0
mean,2000.0,4.0,2269.0,6.0,254.0,4287.0,19.0,30.0
std,14.0,1.0,183.0,1.0,23.0,266.0,1.0,3.0
min,1975.0,2.0,1590.0,4.0,171.0,3320.0,14.0,20.0
25%,1987.0,3.0,2150.0,6.0,239.0,4110.0,18.0,28.0
50%,2000.0,4.0,2270.0,6.0,254.0,4280.0,19.0,30.0
75%,2012.0,4.0,2390.0,6.0,270.0,4470.0,20.0,32.0
max,2024.0,5.0,3110.0,8.0,334.0,5460.0,24.0,41.0


In [6]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [7]:
df.horsepower = df.horsepower.fillna(value=0)
df.acceleration = df.acceleration.fillna(value=0)

In [8]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

In [12]:
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [13]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [14]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [15]:
y_train = df_train.fuel_efficiency_mpg
y_val = df_val.fuel_efficiency_mpg
y_test = df_test.fuel_efficiency_mpg

In [16]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# Question 1

In [26]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

In [56]:
train_dicts = df_train.fillna(0).to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)


In [28]:
y_train

0       30.8
1       29.7
2       27.6
3       33.7
4       29.3
        ... 
5995    29.3
5996    28.8
5997    30.7
5998    32.1
5999    29.6
Name: fuel_efficiency_mpg, Length: 6000, dtype: float64

In [29]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [30]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



# Question 2

In [35]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_pred = rf.predict(X_val)
# y_pred
# y_pred = rf.predict(X_val)[:, 1]
# auc = roc_auc_score(y_val, y_pred)
print(rmse(y_val, y_pred))

1.837054054730018


# Question 3

In [37]:
for n in [10, 50, 100, 150, 200]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    print(f"rmse for {n}: {rmse(y_val, y_pred)}")

rmse for 10: 1.837054054730018
rmse for 50: 1.7715666428333987
rmse for 100: 1.7681362324492986
rmse for 150: 1.7688040324216567
rmse for 200: 1.7682935626261276


In [38]:
def rmse_custom(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [40]:
for n in [10, 50, 100, 150, 200]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    print(f"rmse custom for {n}: {round(rmse_custom(y_val, y_pred), 3)}")

rmse custom for 10: 1.837
rmse custom for 50: 1.772
rmse custom for 100: 1.768
rmse custom for 150: 1.769
rmse custom for 200: 1.768


# Question 4

In [45]:
best_rmse = 2
best_depth = 1
for depth in [10, 15, 20, 25]:
    rmse_list = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(max_depth=depth, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse_list.append(rmse(y_val, y_pred))
    rmse_mean = np.mean(rmse_list)   
    if best_rmse > rmse_mean:
        best_rmse = rmse_mean
        best_depth = depth
print(f"best rmse {best_rmse} at depth {best_depth}")

best rmse 1.7554047863550186 at depth 10


# Question 5

In [58]:
rf = RandomForestRegressor(max_depth=20, n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
names = dv.feature_names_
most_important = np.argmax(importances)
specific_names = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
for sn in specific_names:
    idx = 0
    for i in range(len(names)):
        if names[i] == sn:
            idx = i
            break
    importance = importances[idx]
    print(f"{sn} importance {importance}")


vehicle_weight importance 0.20387857204555604
horsepower importance 0.07801732200517682
acceleration importance 0.05520111946946774
engine_displacement importance 0.06100490942209099


 # Question 6

In [59]:
import xgboost as xgb

In [60]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [61]:
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [65]:
xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}
model = xgb.train(xgb_params, dtrain, num_boost_round=100)

In [66]:
y_pred = model.predict(dval)
rmse(y_val, y_pred)

1.724810059827976